<a href="https://colab.research.google.com/github/sri-wahyuni10/Inter-flyrank/blob/main/work%20/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Week 3: Data Contract & Feature Leakage Experiment

## 1) & 2) The Contract (In Plain Words)

* **What one row means (The Grain):** One row represents a unique combination of a specific page on a specific calendar date (`page` + `date`).
* **Which table(s) you'll use:** FlyRank's monthly dataset loaded from Hugging Face via DuckDB Parquet reader (`hf://datasets/FlyRank/internship-warehouse/data/month=2026-03/*.parquet`).
* **Which time window:** Feature exploration and engineering were conducted in the middle of the panel, namely March 2026 (`month=2026-03`). June 2026 was intentionally omitted as a pure test window.
* **What you'd predict or rank (Label or Proxy):** Using the **CTR/Engagement Opportunity Scoring** approach. The predicted target is whether a page with sufficient impressions will experience click performance below market expectations (under-capturing CTR) in the future time window (`is_ctr_opportunity`).
* **One thing deliberately excluded:** Pages with fewer than 10 total daily impressions are explored and discarded (`impressions < 10`), as CTR fluctuations at low volumes are random (*noise*) and not a valid performance signal.

In [1]:
import os
import duckdb
import pandas as pd
from google.colab import userdata

In [2]:
# Google Colab akan otomatis mengambil token dari panel gembok secara aman
hf_token = userdata.get("HF_TOKEN")

if hf_token:
    print("Token Successfully Loaded from Secrets Securely!")
else:
    print("Token Failed to Load. Make sure the Secret name is HF_TOKEN and access is activated.")

Token Successfully Loaded from Secrets Securely!


In [3]:
# 1. Inisialisasi koneksi DuckDB
con = duckdb.connect()

# 2. Ambil token dari panel Secrets Colab
hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise ValueError("Make sure you have set the Secrets 'HF_TOKEN' in Google Colab!")

# 3. Muat ekstensi httpfs untuk DuckDB v1.3.x
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# 4. Konfigurasi HTTP Header Auth menggunakan parameter yang tepat untuk v1.3.x
con.execute(f"""
    CREATE SECRET http_auth (
        TYPE http,
        EXTRA_HTTP_HEADERS MAP {{'Authorization': 'Bearer {hf_token}'}}
    );
""")
print("Header Authentication successfully configured via HTTP Secrets Manager!")

# Jalur dataset bulan tengah panel (Maret 2026)
dataset_url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/**/*.parquet"

Header Authentication successfully configured via HTTP Secrets Manager!


In [7]:
df_check = con.execute(f"SELECT * FROM read_parquet('{dataset_url}') LIMIT 10").df()
available_cols = df_check.columns.tolist()

# Otomatis deteksi nama kolom halaman dan tanggal yang valid di environment Anda
col_page = "url" if "url" in available_cols else ("content_id" if "content_id" in available_cols else available_cols[0])
col_date = "date" if "date" in available_cols else ("report_date" if "report_date" in available_cols else "month")
col_metric = "ga4_pageviews" if "ga4_pageviews" in available_cols else available_cols[1]

print(f"The system detects your main column -> Page: '{col_page}', date: '{col_date}', Metric: '{col_metric}'\n")

print("\n======================================================================")
print("QUERY 1: VERIFIKASI GRAIN (Page + Date Uniqueness)")
print("========================================================================")
q1 =  f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT {col_page} || {col_date}) as unique_grain_rows
FROM read_parquet('{dataset_url}')
"""
df_q1 = con.execute(q1).df()
display(df_q1)


The system detects your main column -> Page: 'report_date', date: 'report_date', Metric: 'ga4_pageviews'


QUERY 1: VERIFIKASI GRAIN (Page + Date Uniqueness)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_grain_rows
0,9841378,31


In [8]:
print("\n========================================================================")
print("QUERY 2: ROW COUNT AND DATE SPAN")
print("========================================================================")
q2 =f"""
SELECT
    COUNT(*) as row_count,
    MIN(report_date) as start_date,
    MAX(report_date) as end_date
FROM '{dataset_url}'
"""
df_q2 = con.execute(q2).df()
display(df_q2)


QUERY 2: ROW COUNT AND DATE SPAN


,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [9]:
print("\n========================================================================")
print("QUERY 3: AVAILABILITY FILTERS (IS TRUE)")
print("========================================================================")
q3 = f"""
SELECT
    COUNT(*) as surviving_rows
FROM '{dataset_url}'
WHERE (gsc_impressions >= 10 AND gsc_clicks IS NOT NULL) IS TRUE
"""
df_q3 = con.execute(q3).df()
display(df_q3)



QUERY 3: AVAILABILITY FILTERS (IS TRUE)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,surviving_rows
0,2147529


In [10]:
print("\n========================================================================")
print("BUILDING 5 FEATURES + THE LEAKAGE TRAP EXPERIMENT")
print("========================================================================")
q_features = f"""
SELECT
    {col_page} as page_identifier,
    report_date,
    gsc_clicks as raw_clicks,
    gsc_impressions as raw_impressions,
    gsc_sum_position as avg_position,
    (gsc_clicks / NULLIF(gsc_impressions, 0)) as historical_ctr,
    CASE WHEN gsc_sum_position <= 3 THEN 1 ELSE 0 END as is_top_three,
    -- [THE TRAP]: Sengaja memasukkan kolom bocor yang ditarik dari label/masa depan
    (gsc_clicks * 1.0) as leaked_label_derived
FROM '{dataset_url}'
WHERE (gsc_impressions >= 10) IS TRUE
LIMIT 5
"""

df_features = con.execute(q_features).df()

print("Dataframe with Active Leak (Trap Active):")
display(df_features)

# MENGEKSEKUSI PELAJARAN DATA LEAKAGE: Hapus kolom bocor agar mendapatkan skor jujur
df_features_honest = df_features.drop(columns=['leaked_label_derived'])
print("\nDataframe Jujur (Trap Successfully Removed):")
display(df_features_honest)


BUILDING 5 FEATURES + THE LEAKAGE TRAP EXPERIMENT
Dataframe with Active Leak (Trap Active):


,page_identifier,report_date,raw_clicks,raw_impressions,avg_position,historical_ctr,is_top_three,leaked_label_derived
0,2026-03-01,2026-03-01,0,20,67,0.000000,0,0.0
1,2026-03-01,2026-03-01,1,125,616,0.008000,0,1.0
2,2026-03-01,2026-03-01,0,11,25,0.000000,0,0.0
3,2026-03-01,2026-03-01,1,239,1756,0.004184,0,1.0
4,2026-03-01,2026-03-01,0,191,1496,0.000000,0,0.0



Dataframe Jujur (Trap Successfully Removed):


,page_identifier,report_date,raw_clicks,raw_impressions,avg_position,historical_ctr,is_top_three
0,2026-03-01,2026-03-01,0,20,67,0.000000,0
1,2026-03-01,2026-03-01,1,125,616,0.008000,0
2,2026-03-01,2026-03-01,0,11,25,0.000000,0
3,2026-03-01,2026-03-01,1,239,1756,0.004184,0
4,2026-03-01,2026-03-01,0,191,1496,0.000000,0


### "Available When?" Feature Lines:
* `raw_clicks`: *knowable at the moment of decision* because today's click interaction data is recorded immediately at the close of the current calendar day.
* `raw_impressions`: *knowable at the moment of decision* because pageview data is recorded immediately by the search log system on the same day.
* `avg_position`: *knowable at the moment of decision* because the average rank is calculated and locked at the end of the day.
* `historical_ctr`: *knowable at the moment of decision* because it is derived deterministically from the current day's internal metrics.
* `is_top_three`: *knowable at the moment of decision* because it is a purely binary logic transformation of today's ranking position.

---

## 4) Named Limitation of Your Slice

**Named Limitation:** This daily performance data slice is susceptible to *Data Freshness Delay* issues. Google Search Console data streamed to BigQuery via daily sync has a built-in lag of approximately 24 to 48 hours from Google. Therefore, the "today" feature actually reflects the reality of performance two days ago, preventing the system from making *real-time* corrective interventions per second.

---